<a href="https://colab.research.google.com/github/Akpati-Lucan/algoverse-research/blob/master/Diffusion_BackProp_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

File to test tthe Diffusion Backprop

Imports

In [ ]:
import torch
import matplotlib.pyplot as plt
import deepinv as dinv
from torchvision import datasets, transforms

ModuleNotFoundError: No module named 'deepinv'

Device

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

Recreate the diffusion schedule

In [ ]:
timesteps = 1000

beta_start = 1e-4
beta_end = 0.02

betas = torch.linspace(beta_start, beta_end, timesteps).to(device)

alphas = 1.0 - betas

alphas_cumprod = torch.cumprod(alphas, dim=0)

sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)

sqrt_one_minus_alphas_cumprod = torch.sqrt(1 - alphas_cumprod)

Load the test dataset

In [ ]:
transform = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize((0.0,), (1.0,))
])

test_loader = torch.utils.data.DataLoader(
    datasets.MNIST(
        "./data",
        train=False,
        download=True,
        transform=transform
    ),
    batch_size=8,
    shuffle=True
)

oad your trained model

In [ ]:

model = dinv.models.DiffUNet(
    in_channels=1,
    out_channels=1,
    pretrained=None
).to(device)

model.load_state_dict(
    torch.load("trained_diffusion_model.pth", map_location=device)
)

model.eval()

print("Model Loaded")

Get a batch of test images

In [ ]:
images, labels = next(iter(test_loader))

images = images.to(device)

print(images.shape)

Add random Gaussian noise

In [ ]:
noise = torch.randn_like(images)

t = torch.randint(
    0,
    timesteps,
    (images.size(0),),
    device=device
)

noisy_images = (
    sqrt_alphas_cumprod[t,None,None,None] * images +
    sqrt_one_minus_alphas_cumprod[t,None,None,None] * noise
)

NameError: name 'images' is not defined

Predict the noise

In [ ]:
with torch.no_grad():

    predicted_noise = model(
        noisy_images,
        t,
        type_t="timestep"
    )

Compute the test loss

In [ ]:
criterion = torch.nn.MSELoss()

loss = criterion(predicted_noise, noise)

print("Noise Prediction Loss:", loss.item())

Cell 10 – Recover the original image

The DDPM forward process is

xt​=αˉtx0+1−αˉtϵ.

Rearranging gives

x^0=αˉtxt−1−αˉtϵ^

In [ ]:
recovered = (
    noisy_images -
    sqrt_one_minus_alphas_cumprod[t,None,None,None]
    * predicted_noise
) / sqrt_alphas_cumprod[t,None,None,None]

NameError: name 'noisy_images' is not defined

Display the results

In [ ]:
fig, axes = plt.subplots(3,8, figsize=(16,6))

for i in range(8):

    axes[0,i].imshow(
        images[i,0].cpu(),
        cmap="gray"
    )
    axes[0,i].set_title("Original")
    axes[0,i].axis("off")

    axes[1,i].imshow(
        noisy_images[i,0].cpu(),
        cmap="gray"
    )
    axes[1,i].set_title("Noisy")
    axes[1,i].axis("off")

    axes[2,i].imshow(
        recovered[i,0].cpu(),
        cmap="gray"
    )
    axes[2,i].set_title("Recovered")
    axes[2,i].axis("off")

plt.tight_layout()
plt.show()

Compare the true and predicted noise

In [ ]:
fig, axes = plt.subplots(2,8, figsize=(16,4))

for i in range(8):

    axes[0,i].imshow(
        noise[i,0].cpu(),
        cmap="gray"
    )
    axes[0,i].set_title("True")
    axes[0,i].axis("off")

    axes[1,i].imshow(
        predicted_noise[i,0].cpu(),
        cmap="gray"
    )
    axes[1,i].set_title("Pred")
    axes[1,i].axis("off")

plt.tight_layout()
plt.show()